# 02 — Batch Video Inference on Ray + vLLM (Serverless GPU)

Runs Qwen2.5-VL over every video in the reference table with a two-stage Ray Data pipeline, and writes the model's analysis to a Delta table (default `video_inferences`) for notebook 03.
1. **Prepare (CPU):** decode each video with PyAV and sample frames at about 1 fps (at most `MAX_FRAMES`), keeping the real timestamp of each frame.
2. **Infer (GPU):** vLLM loads the registered weights once and runs batches of videos through `generate()`.

**Compute:** Serverless GPU. Under **Connect**, choose **Serverless GPU**; in the **Environment** panel, set **Accelerator** to **H100** (1 GPU) and the base environment to **Databricks AI** (v6), then apply. The AI environment provides torch, vLLM, Ray and transformers; only PyAV is installed here.

Differences from the classic version (`3_Video_Batch_Inference_Ray_Core`):
* **Correct timestamps.** Each prompt passes vLLM `video_metadata` with the actual sampling rate (frames / duration), and the prompt states the video length and the time between frames. The classic notebook always passed `fps=1`, even when about 30 frames were spread across the whole video.
* **Real batching.** Each `generate()` call receives a batch of videos, not one.
* **No system ffmpeg.** PyAV (pip) replaces `apt-get install ffmpeg`.
* **One model download.** The weights are staged once on the driver instead of once per actor.
* **Output path.** Results go to Parquet in a UC Volume, then become a Delta table through Spark (`write_databricks_table` is classic-only).

**Troubleshooting**
* **Do not `%pip install` vLLM, torch, transformers, or Ray.** If you did, **New Session** does not undo it (the notebook environment is rebuilt with the same packages). Uninstall them, or reset the environment in the **Environment** panel, so the AI v6 versions are used again.
* **`FileNotFoundError: 'ninja'` or an engine startup failure:** the Ray pipeline cell already puts the AI environment's `bin/` on `PATH` for the workers and turns off the FlashInfer sampler (`VLLM_USE_FLASHINFER_SAMPLER=0`). Keep those lines.
* **`ValueError: The plasma directory ... /local_disk0/tmp does not exist`:** the cell sets `RAY_TMPDIR=/tmp/ray` because `/local_disk0` is not writable on serverless.
* **Quick test:** set `MAX_VIDEOS` to 3. A full run of 49 videos takes about 3 minutes on 1×H100 once the weights are staged (staging takes about 4 minutes the first time).

In [0]:
# The Databricks AI environment (v6) already ships torch, vLLM, Ray and transformers built together.
# Do not pip-install those: it replaces torch and breaks the preinstalled compiled extensions (flash-attn).
%pip install -q "av==14.0.1"
dbutils.library.restartPython()

In [0]:
import torch, vllm, ray, transformers, av

assert torch.cuda.is_available(), "No GPU visible. Attach Serverless GPU (H100) before running this notebook."
NUM_GPUS = torch.cuda.device_count()
print(f"GPUs: {NUM_GPUS} x {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__} | vllm {vllm.__version__} | ray {ray.__version__} | transformers {transformers.__version__} | av {av.__version__}")
assert int(transformers.__version__.split(".")[0]) >= 5, "Expected the Databricks AI v6 environment (transformers 5.x); see the Overview cell."

In [0]:
dbutils.widgets.text("CATALOG", "main", label="CATALOG")
dbutils.widgets.text("SCHEMA", "vlm_video", label="SCHEMA")  # same as notebook 00
dbutils.widgets.text("VOLUME", "transcribe-video", label="VOLUME")
dbutils.widgets.text("MODEL_NAME", "qwen2_5_vl-7b", label="MODEL_NAME")
dbutils.widgets.text("MODEL_ALIAS", "production", label="MODEL_ALIAS")
dbutils.widgets.text("INPUT_TABLE", "videos_file_reference", label="INPUT_TABLE")
dbutils.widgets.text("OUTPUT_TABLE", "video_inferences", label="OUTPUT_TABLE")
dbutils.widgets.text("CATEGORY_FILTER", "shoplifting", label="CATEGORY_FILTER")
dbutils.widgets.text("MAX_DURATION_SECONDS", "1200", label="MAX_DURATION_SECONDS")
# 0 = all matching videos. Set to e.g. 3 for a quick smoke test.
dbutils.widgets.text("MAX_VIDEOS", "0", label="MAX_VIDEOS")
dbutils.widgets.text("TARGET_FPS", "1", label="TARGET_FPS")
dbutils.widgets.text("MAX_FRAMES", "64", label="MAX_FRAMES")
# 7B: TP=1, no quantization. 32B on 1xH100: TP=1 + fp8. 32B on a multi-GPU node: TP=2, no quantization.
dbutils.widgets.text("TENSOR_PARALLEL", "1", label="TENSOR_PARALLEL")
dbutils.widgets.dropdown("QUANTIZATION", "none", ["none", "fp8"], label="QUANTIZATION")

CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")
VOLUME = dbutils.widgets.get("VOLUME")
MODEL_NAME = dbutils.widgets.get("MODEL_NAME")
MODEL_ALIAS = dbutils.widgets.get("MODEL_ALIAS")
INPUT_TABLE = dbutils.widgets.get("INPUT_TABLE")
OUTPUT_TABLE = dbutils.widgets.get("OUTPUT_TABLE")
CATEGORY_FILTER = dbutils.widgets.get("CATEGORY_FILTER")
MAX_DURATION_SECONDS = float(dbutils.widgets.get("MAX_DURATION_SECONDS"))
MAX_VIDEOS = int(dbutils.widgets.get("MAX_VIDEOS"))
TARGET_FPS = float(dbutils.widgets.get("TARGET_FPS"))
MAX_FRAMES = int(dbutils.widgets.get("MAX_FRAMES"))
TENSOR_PARALLEL = int(dbutils.widgets.get("TENSOR_PARALLEL"))
QUANTIZATION = dbutils.widgets.get("QUANTIZATION")

UC_MODEL = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
INPUT_FQN = f"{CATALOG}.{SCHEMA}.{INPUT_TABLE}"
OUTPUT_FQN = f"{CATALOG}.{SCHEMA}.{OUTPUT_TABLE}"
PARQUET_OUTPUT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/inference_output/{OUTPUT_TABLE}"
print(f"{UC_MODEL}@{MODEL_ALIAS}: {INPUT_FQN} -> {OUTPUT_FQN}")

In [0]:
import os, time, mlflow
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")
mv = MlflowClient().get_model_version_by_alias(UC_MODEL, MODEL_ALIAS)

# /local_disk0 is not writable on serverless; /tmp is local disk.
stage_root = f"/tmp/models/{MODEL_NAME}_v{mv.version}"
MODEL_PATH = f"{stage_root}/artifacts/weights"

if not os.path.exists(f"{MODEL_PATH}/model.safetensors.index.json"):
    t0 = time.time()
    MODEL_PATH = mlflow.artifacts.download_artifacts(
        artifact_uri=f"models:/{UC_MODEL}@{MODEL_ALIAS}/artifacts/weights",
        dst_path=stage_root,
    )
    print(f"Downloaded in {(time.time() - t0) / 60:.1f} min")

size_gb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(MODEL_PATH) for f in fs) / 1e9
print(f"{UC_MODEL} v{mv.version} staged at {MODEL_PATH} ({size_gb:.1f} GB)")

In [0]:
import pyspark.sql.functions as F

videos_df = (
    spark.table(INPUT_FQN)
    .withColumn("category", F.element_at(F.split("file_path", "/"), -2))
    .filter((F.col("category") == CATEGORY_FILTER) & (F.col("duration_seconds") <= MAX_DURATION_SECONDS))
    .select("path", "name", "file_path", "size", "duration_seconds", "category")
    .orderBy("name")
)
if MAX_VIDEOS > 0:
    videos_df = videos_df.limit(MAX_VIDEOS)

rows = videos_df.toPandas().to_dict("records")
assert rows, f"No videos matched category={CATEGORY_FILTER}, duration<={MAX_DURATION_SECONDS}"
print(f"{len(rows)} videos, {sum(r['size'] for r in rows) / 1e9:.2f} GB, {sum(r['duration_seconds'] for r in rows) / 60:.0f} min of footage")

In [0]:
import json
import numpy as np

QUERY = """
Analyze the surveillance video and determine if any shoplifting activity occurs. Focus on suspicious behaviors such as: hiding merchandise in clothing or bags, intentionally bypassing the checkout, switching price tags, or attempting to distract staff.  If shoplifting is detected, describe what happened, when it occurred (with timestamps), and which person was involved. Be specific and detailed in your explanation.
"""


def sample_frames(path, target_fps=1.0, max_frames=64, min_frames=4, max_side=448):
    """Uniformly sample frames at ~target_fps, capped at max_frames.

    The frame count is kept even (Qwen2.5-VL groups frames in pairs). Returns
    (frames [T,H,W,3] uint8, effective fps, per-frame timestamps in seconds, duration).
    """
    import av

    with av.open(path) as container:
        stream = container.streams.video[0]
        stream.thread_type = "AUTO"
        if stream.duration is not None:
            duration = float(stream.duration * stream.time_base)
        else:
            duration = container.duration / 1e6
        n = max(min_frames, min(max_frames, int(round(duration * target_fps))))
        n -= n % 2
        targets = [(i + 0.5) * duration / n for i in range(n)]
        w, h = stream.codec_context.width, stream.codec_context.height
        scale = min(1.0, max_side / max(w, h))
        out_w, out_h = int(w * scale) // 2 * 2, int(h * scale) // 2 * 2

        frames, stamps, k = [], [], 0
        for frame in container.decode(stream):
            if frame.time is None:
                continue
            while k < n and frame.time >= targets[k]:
                frames.append(frame.to_ndarray(format="rgb24", width=out_w, height=out_h))
                stamps.append(round(float(frame.time), 2))
                k += 1
            if k >= n:
                break

    if len(frames) < min_frames:
        raise ValueError(f"decoded only {len(frames)} frames")
    if len(frames) % 2:
        frames, stamps = frames[:-1], stamps[:-1]
    return np.stack(frames), len(frames) / duration, stamps, duration


def build_prompt(question, n_frames, duration, eff_fps):
    timing = (
        f"The video is {duration:.0f} seconds long and is shown as {n_frames} frames, "
        f"one every {1 / eff_fps:.1f} seconds. Give timestamps as seconds from the start of the video "
        f"(0 to {duration:.0f}), not the clock shown on screen. Summarize the key events only; do not list every frame."
    )
    return (
        "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
        "<|im_start|>user\n<|vision_start|><|video_pad|><|vision_end|>"
        f"{timing}\n{question}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


class PrepareVideo:
    """CPU stage: decode and sample frames, build the prompt. Failures are flagged, not raised."""

    def __init__(self, query, target_fps, max_frames):
        self.query, self.target_fps, self.max_frames = query, target_fps, max_frames

    def __call__(self, row):
        try:
            frames, eff_fps, stamps, duration = sample_frames(row["file_path"], self.target_fps, self.max_frames)
            row.update(
                frames=frames, decode_ok=True, decode_error="",
                num_frames=len(stamps), sample_fps=float(eff_fps), frame_timestamps=json.dumps(stamps),
                prompt=build_prompt(self.query, len(stamps), duration, eff_fps),
            )
        except Exception as e:
            row.update(
                frames=np.zeros((0, 1, 1, 3), dtype=np.uint8), decode_ok=False, decode_error=str(e)[:500],
                num_frames=0, sample_fps=0.0, frame_timestamps="[]", prompt="",
            )
        return row


class VLMPredictor:
    """GPU stage: one vLLM engine per actor; each call runs a whole batch through generate()."""

    def __init__(self, model_path, engine_kwargs, sampling_kwargs):
        from vllm import LLM, SamplingParams

        self.llm = LLM(model=model_path, **engine_kwargs)
        self.sampling_params = SamplingParams(**sampling_kwargs)

    def __call__(self, batch):
        prompts = []
        for p, f, fps in zip(batch["prompt"], batch["frames"], batch["sample_fps"]):
            n = len(f)
            prompts.append({
                "prompt": p,
                "multi_modal_data": {"video": np.asarray(f)},
                # Tells Qwen2.5-VL the real time between frames (sets its temporal position ids).
                # transformers 5.x derives this from video metadata; without it, it assumes 24 fps.
                "mm_processor_kwargs": {
                    "video_metadata": [{"fps": float(fps), "total_num_frames": n, "frames_indices": list(range(n)), "duration": n / float(fps)}]
                },
            })
        outputs = self.llm.generate(prompts, self.sampling_params, use_tqdm=False)
        batch.pop("frames")
        batch["generated_text"] = np.array([o.outputs[0].text for o in outputs], dtype=object)
        batch["finish_reason"] = np.array([o.outputs[0].finish_reason for o in outputs], dtype=object)
        batch["prompt_tokens"] = np.array([len(o.prompt_token_ids) for o in outputs])
        batch["output_tokens"] = np.array([len(o.outputs[0].token_ids) for o in outputs])
        return batch

In [0]:
# 7B in bf16 is ~16.6 GB, which leaves most of an 80 GB H100 for the KV cache.
# 32B in bf16 is ~66 GB: use fp8 (~33 GB) on one H100, or TENSOR_PARALLEL=2.
# A 64-frame 320x240 clip is ~3.5K visual tokens, so 16K context leaves headroom.
assert TENSOR_PARALLEL <= NUM_GPUS, f"TENSOR_PARALLEL={TENSOR_PARALLEL} but only {NUM_GPUS} GPU(s) attached"
ENGINE_KWARGS = {
    "dtype": "bfloat16",
    "tensor_parallel_size": TENSOR_PARALLEL,
    "max_model_len": 16384,
    "max_num_seqs": 8,
    "gpu_memory_utilization": 0.90,
    "limit_mm_per_prompt": {"image": 0, "video": 1},
    "mm_processor_cache_gb": 0,  # every video is unique; replaces disable_mm_preprocessor_cache
}
if QUANTIZATION != "none":
    ENGINE_KWARGS["quantization"] = QUANTIZATION  # online fp8: weights quantized at load time
if TENSOR_PARALLEL > 1:
    # Shard across the GPUs Ray assigned to this actor with local processes, rather than
    # having vLLM create its own Ray placement group from inside a Ray Data actor.
    ENGINE_KWARGS["distributed_executor_backend"] = "mp"

SAMPLING_KWARGS = {
    "temperature": 0.1,
    "top_p": 1.0,
    "repetition_penalty": 1.05,
    "max_tokens": 1024,  # 2500 let a few outputs list events until the cap (or loop on one token)
}

BATCH_SIZE = 8                           # videos per generate() call
GPU_ACTORS = max(1, NUM_GPUS // ENGINE_KWARGS["tensor_parallel_size"])

In [0]:
import ray, sys

# Serverless sets RAY_TMPDIR=/local_disk0/tmp, which is not writable; Ray fails at init without this.
os.environ["RAY_TMPDIR"] = "/tmp/ray"
os.makedirs(os.environ["RAY_TMPDIR"], exist_ok=True)
# The AI environment's bin/ (with ninja, used by FlashInfer JIT builds) is not on PATH by default.
env_bin = os.path.dirname(os.path.realpath(sys.executable)) if "databricks-environments" in os.path.realpath(sys.executable) else "/opt/databricks-environments/databricks-ai/bin"
WORKER_ENV = {
    "PATH": f"{env_bin}:{os.environ['PATH']}",
    "VLLM_USE_FLASHINFER_SAMPLER": "0",  # PyTorch-native top-k/top-p; avoids a JIT kernel build at startup
}
os.environ.update(WORKER_ENV)
if ray.is_initialized():
    ray.shutdown()  # workers only pick up env changes when Ray starts
ray.init(include_dashboard=False, runtime_env={"env_vars": WORKER_ENV})
print(ray.cluster_resources())

t0 = time.time()
ds = (
    ray.data.from_items(rows)
    .repartition(len(rows))
    .map(
        PrepareVideo,
        fn_constructor_kwargs={"query": QUERY, "target_fps": TARGET_FPS, "max_frames": MAX_FRAMES},
        concurrency=(1, 8),
        num_cpus=1,
    )
    .filter(lambda r: r["decode_ok"])
    .map_batches(
        VLMPredictor,
        fn_constructor_kwargs={"model_path": MODEL_PATH, "engine_kwargs": ENGINE_KWARGS, "sampling_kwargs": SAMPLING_KWARGS},
        batch_size=BATCH_SIZE,
        concurrency=GPU_ACTORS,
        num_gpus=ENGINE_KWARGS["tensor_parallel_size"],
    )
)
ds.write_parquet(PARQUET_OUTPUT_PATH, mode="overwrite")
print(f"Inference for {len(rows)} videos finished in {(time.time() - t0) / 60:.1f} min -> {PARQUET_OUTPUT_PATH}")

In [0]:
results_df = spark.read.parquet(PARQUET_OUTPUT_PATH)
(
    results_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_FQN)
)

out = spark.table(OUTPUT_FQN)
n_out = out.count()
print(f"{n_out} of {len(rows)} videos written to {OUTPUT_FQN}")
if n_out < len(rows):
    done = {r.file_path for r in out.select("file_path").collect()}
    print("Not in output (decode failure):", [r["name"] for r in rows if r["file_path"] not in done])

In [0]:
display(
    out.agg(
        F.count("*").alias("videos"),
        F.sum(F.when(F.length("generated_text") == 0, 1).otherwise(0)).alias("empty_outputs"),
        F.sum(F.when(F.col("finish_reason") != "stop", 1).otherwise(0)).alias("not_stopped"),
        F.round(F.avg("num_frames"), 1).alias("avg_frames"),
        F.max("prompt_tokens").alias("max_prompt_tokens"),
        F.round(F.avg("output_tokens"), 0).alias("avg_output_tokens"),
    )
)
display(out.select("name", "duration_seconds", "num_frames", "sample_fps", "finish_reason", "generated_text").orderBy("name").limit(5))